In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from scrapers import barbora, rimi
from scrapers.common import save_rows

In [ ]:
rows = barbora.scrape("mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena", max_pages=3)
print(save_rows(rows, "barbora", "fresh-meat"))

In [ ]:
import requests
from scrapers.common import HEADERS

url = "https://barbora.lt/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena"

# A: the headers from common.py (with Accept-Language)
a = requests.get(url, headers=HEADERS, params={"page": 1}, timeout=20)
print("A:", a.status_code, a.url, len(a.text), "has list:", "window.b_productList" in a.text)

# B: only the User-Agent, like the working notebook
b = requests.get(url, headers={"User-Agent": HEADERS["User-Agent"]}, params={"page": 1}, timeout=20)
print("B:", b.status_code, b.url, len(b.text), "has list:", "window.b_productList" in b.text)

In [ ]:
import re, requests
from scrapers.common import HEADERS

url = "https://barbora.lt/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena"
r = requests.get(url, headers=HEADERS, params={"page": 1}, timeout=20)
html = r.text

print("redirects:", [(h.status_code, h.headers.get("location")) for h in r.history])
print("final url:", r.url)
print("window vars:", re.findall(r"window\.\w+\s*=", html)[:40])
print("__NEXT_DATA__:", "__NEXT_DATA__" in html, "| ld+json:", html.count("application/ld+json"))
for needle in ['"Url":"', "productList", "b_productList", "sprandin", "cnstrc", "comparative_unit_price"]:
    print(f"{needle!r}:", html.count(needle))

# the 5 biggest <script> blocks, with the first 150 characters of each
scripts = re.findall(r"<script[^>]*>(.*?)</script>", html, re.S)
for s in sorted(scripts, key=len, reverse=True)[:5]:
    print(len(s), "|", s[:150].replace("\n", " "))

In [ ]:
import json, re, requests
from scrapers.common import HEADERS

url = "https://barbora.lt/c/LVOK/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena"
html = requests.get(url, headers=HEADERS, params={"page": 1}, timeout=20).text

marker = "window.__remixContext ="
i = html.find(marker) + len(marker)
while html[i].isspace():
    i += 1
ctx, _ = json.JSONDecoder().raw_decode(html, i)
print("top-level keys:", list(ctx.keys()))

def shape(o, path="", depth=0, max_depth=6):
    """Print the nested structure: dict keys and list lengths."""
    if depth > max_depth:
        return
    if isinstance(o, dict):
        for k, v in list(o.items())[:12]:
            kind = f"list[{len(v)}]" if isinstance(v, list) else type(v).__name__
            print(f"{'  '*depth}{k}: {kind}")
            shape(v, f"{path}/{k}", depth + 1, max_depth)
    elif isinstance(o, list) and o:
        shape(o[0], path + "[0]", depth, max_depth)

shape(ctx["state"]["loaderData"])

In [ ]:
def find_lists(o, path=""):
    if isinstance(o, list) and len(o) > 10 and isinstance(o[0], dict):
        print(path, "-> list of", len(o), "| keys:", list(o[0].keys())[:15])
    if isinstance(o, dict):
        for k, v in o.items():
            find_lists(v, f"{path}/{k}")
    elif isinstance(o, list):
        for n, v in enumerate(o[:1]):
            find_lists(v, f"{path}[{n}]")

find_lists(ctx)

In [ ]:
def find_key(o, key):
    """Return (parent_dict, value) for the first dict that has this key."""
    if isinstance(o, dict):
        if key in o:
            return o, o[key]
        for v in o.values():
            hit = find_key(v, key)
            if hit:
                return hit
    elif isinstance(o, list):
        for v in o:
            hit = find_key(v, key)
            if hit:
                return hit
    return None

parent, inventories = find_key(ctx, "inventories")
print("products on page:", len(inventories))

# 1) everything next to the list: probably total count, page info
print("sibling keys:", list(parent.keys()))
for k, v in parent.items():
    if not isinstance(v, (list, dict)):
        print("  ", k, "=", v)

# 2) one full product record
print(json.dumps(inventories[0], ensure_ascii=False, indent=1)[:3500])